In [ ]:
from project_config import CATALOG_ROOT, RADAR_DOC_ROOT, OUTPUT_ROOT, EVALUATION_SPLIT, WEIGHTED_HAILRU_CHECKPOINT, load_pickle
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from tqdm import tqdm
from cnmaps import get_adm_maps, draw_maps
from shapely.geometry import Point
import cartopy.crs as ccrs
from cnmaps import draw_map


In [ ]:
setname = EVALUATION_SPLIT
data_dic = load_pickle(CATALOG_ROOT / setname / 'data_dic_nwp.pkl')
mask_dic = load_pickle(CATALOG_ROOT / setname / 'mask_dic_all.pkl')
train_dic_matrix = load_pickle(CATALOG_ROOT / setname / 'train_dic_matrix.pkl')


In [ ]:
class SpatialAttention(nn.Module):

    def __init__(self):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, 7, 1, 3)
        self.act = nn.Sigmoid()

    def forward(self, x):
        maxpool = torch.max(x, dim=1, keepdim=True)[0]
        avgpool = torch.mean(x, dim=1, keepdim=True)
        SA = self.act(self.conv(torch.cat((maxpool, avgpool), dim=1)))
        return SA * x

class ChannelAttention(nn.Module):

    def __init__(self, dim):
        super(ChannelAttention, self).__init__()
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.maxpool = nn.AdaptiveMaxPool2d(1)
        self.conv_shared = nn.Sequential(nn.Conv2d(dim, dim // 16, 1, 1), nn.LeakyReLU(), nn.Conv2d(dim // 16, dim, 1, 1))
        self.act = nn.Sigmoid()

    def forward(self, x):
        maxpool = self.conv_shared(self.maxpool(x))
        avgpool = self.conv_shared(self.avgpool(x))
        CA = self.act(maxpool + avgpool)
        return CA * x

class CBAM(nn.Module):

    def __init__(self, dim):
        super(CBAM, self).__init__()
        self.CA = ChannelAttention(dim)
        self.SA = SpatialAttention()

    def forward(self, x):
        x = self.CA(x)
        x = self.SA(x)
        return x

class IRCBAM(nn.Module):

    def __init__(self, inchannels):
        super(IRCBAM, self).__init__()
        self.c = inchannels
        self.act = nn.LeakyReLU()
        self.branch1 = nn.Sequential(nn.Conv2d(self.c, self.c // 4, 1, 1, 'same'), nn.BatchNorm2d(self.c // 4), self.act)
        self.branch2 = nn.Sequential(nn.Conv2d(self.c, self.c // 4, 1, 1, 'same'), nn.BatchNorm2d(self.c // 4), self.act, nn.Conv2d(self.c // 4, self.c // 4, 3, 1, 'same'), nn.BatchNorm2d(self.c // 4), self.act)
        self.branch3 = nn.Sequential(nn.Conv2d(self.c, self.c // 4, 1, 1, 'same'), nn.BatchNorm2d(self.c // 4), self.act, nn.Conv2d(self.c // 4, self.c // 4, 3, 1, 'same'), nn.BatchNorm2d(self.c // 4), self.act, nn.Conv2d(self.c // 4, self.c // 4, 3, 1, 'same'), nn.BatchNorm2d(self.c // 4), self.act)
        self.branch4 = nn.Sequential(nn.AvgPool2d(3, 1, 1), nn.Conv2d(self.c, self.c // 4, 1, 1, 'same'), nn.BatchNorm2d(self.c // 4), self.act)
        self.CBAM = CBAM(inchannels)

    def forward(self, x):
        identity = x
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)
        cat = torch.cat([b1, b2, b3, b4], dim=1)
        cat = self.CBAM(cat)
        output = 0.3 * cat + identity
        return output

class HRUnet(nn.Module):

    def __init__(self, in_channels, out_channels):
        super(HRUnet, self).__init__()
        chans = [64, 256, 1024]
        ks = 3
        self.act = nn.LeakyReLU(inplace=True)
        self.act_output = nn.Sigmoid()
        self.channel_weights = nn.Parameter(torch.ones(1, in_channels, 1, 1))
        self.start_conv = nn.Sequential(nn.Conv2d(in_channels, chans[0], 2, 1, 0), nn.BatchNorm2d(chans[0]), self.act)
        self.ResBlock_11 = IRCBAM(chans[0])
        self.ResBlock_12 = IRCBAM(chans[0])
        self.DownConv_1 = nn.PixelUnshuffle(2)
        self.ResBlock_21 = IRCBAM(chans[1])
        self.ResBlock_22 = IRCBAM(chans[1])
        self.DownConv_2 = nn.PixelUnshuffle(2)
        self.ResBlock_31 = IRCBAM(chans[2])
        self.ResBlock_32 = IRCBAM(chans[2])
        self.ResBlock_33 = IRCBAM(chans[2])
        self.ResBlock_34 = IRCBAM(chans[2])
        self.UpConv_1 = nn.PixelShuffle(2)
        self.UpConv_next_1 = nn.Sequential(nn.Conv2d(chans[1] * 2, chans[1], ks, 1, 1), nn.BatchNorm2d(chans[1]), self.act)
        self.ResBlock_23 = IRCBAM(chans[1])
        self.ResBlock_24 = IRCBAM(chans[1])
        self.UpConv_2 = nn.PixelShuffle(2)
        self.UpConv_next_2 = nn.Sequential(nn.Conv2d(chans[0] * 2, chans[0], ks, 1, 1), nn.BatchNorm2d(chans[0]), self.act)
        self.ResBlock_13 = IRCBAM(chans[0])
        self.ResBlock_14 = IRCBAM(chans[0])
        self.final_conv = nn.Sequential(nn.Conv2d(chans[0], out_channels, 2, 1, 1), self.act_output)

    def forward(self, x, mask):
        x = x * self.channel_weights
        x = self.start_conv(x)
        x = self.ResBlock_11(x)
        x = self.ResBlock_12(x)
        x_pass_1 = x
        x = self.DownConv_1(x)
        x = self.ResBlock_21(x)
        x = self.ResBlock_22(x)
        x_pass_2 = x
        x = self.DownConv_2(x)
        x = self.ResBlock_31(x)
        x = self.ResBlock_32(x)
        x = self.ResBlock_33(x)
        x = self.ResBlock_34(x)
        x = self.UpConv_1(x)
        x = torch.cat((x, x_pass_2), dim=1)
        x = self.UpConv_next_1(x)
        x = self.ResBlock_23(x)
        x = self.ResBlock_24(x)
        x = self.UpConv_2(x)
        x = torch.cat((x, x_pass_1), dim=1)
        x = self.UpConv_next_2(x)
        x = self.ResBlock_13(x)
        x = self.ResBlock_14(x)
        x = self.final_conv(x)
        x = x * mask
        return x

class Loss(nn.Module):

    def __init__(self, alpha, beta):
        super(Loss, self).__init__()
        self.alpha = alpha
        self.beta = beta

    def forward(self, y_pred, y_true):
        base_temp = torch.square(y_pred - y_true) * ((y_true + self.alpha) / (1 + self.alpha))
        if torch.sum(y_true) > 0:
            temp = base_temp
        else:
            temp = self.beta * base_temp
        divisor = temp.shape[1] * temp.shape[2] * temp.shape[3]
        return torch.sum(temp) / divisor

class LightningModel(pl.LightningModule):

    def __init__(self, alpha, beta):
        super().__init__()
        self.model = HRUnet(102, 20)
        self.criterion = Loss(alpha, beta)

    def forward(self, x, mask):
        return self.model(x, mask)

    def training_step(self, batch, batch_idx):
        inputs, mask, labels = batch
        outputs = self.model(inputs, mask)
        loss = self.criterion(outputs, labels)
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        return loss

    def validation_step(self, batch, batch_idx):
        inputs, mask, labels = batch
        outputs = self.model(inputs, mask)
        val_loss = self.criterion(outputs, labels)
        self.log('val_loss', val_loss, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        return val_loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=0.001, fused=True)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.1)
        return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'interval': 'epoch', 'frequency': 1}}
path = WEIGHTED_HAILRU_CHECKPOINT
checkpoint = torch.load(path, map_location='cpu', weights_only=False)
model = LightningModel(0.1, 10)
model.load_state_dict(checkpoint['state_dict'], strict=True)
model.eval()
model.to('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
sta_dic = {}
data_root = RADAR_DOC_ROOT
file_list = os.listdir(data_root)
for i in range(len(file_list)):
    textid = file_list[i].split('.')[0].split('_')[2]
    sta_file = os.path.join(data_root, file_list[i])
    labelfmt = pd.read_csv(sta_file)
    lonss = labelfmt.iloc[0, 0].split('=')
    lons = round(float(lonss[1]), 4)
    lonee = labelfmt.iloc[2, 0].split('=')
    lone = round(float(lonee[1]), 4)
    latss = labelfmt.iloc[1, 0].split('=')
    lats = round(float(latss[1]), 4)
    latee = labelfmt.iloc[3, 0].split('=')
    late = round(float(latee[1]), 4)
    sta_dic[textid] = [lons, lone, lats, late]


In [ ]:
sta_exists_dic = {}
for setname in ['train', 'validation', 'test']:
    split_matrix = load_pickle(CATALOG_ROOT / setname / 'train_dic_matrix.pkl')
    for case_id in split_matrix:
        stid = case_id.split('_')[-1]
        sta_exists_dic[stid] = 0
sta_ll_dic = {}
for stid in sta_dic:
    if stid in sta_exists_dic:
        lon = round((sta_dic[stid][0] + sta_dic[stid][1]) / 2, 4)
        lat = round((sta_dic[stid][2] + sta_dic[stid][3]) / 2, 4)
        sta_ll_dic[stid] = [lon, lat]


In [ ]:
my_sites = sta_ll_dic
REGION_MAPPING = {'黑龙江': 'Northeast', '吉林': 'Northeast', '辽宁': 'Northeast', '北京': 'North', '天津': 'North', '河北': 'North', '山西': 'North', '内蒙古': 'North', '陕西': 'Northwest', '甘肃': 'Northwest', '青海': 'Northwest', '宁夏': 'Northwest', '新疆': 'Northwest', '上海': 'East', '江苏': 'East', '浙江': 'East', '安徽': 'East', '福建': 'East', '江西': 'East', '山东': 'East', '台湾': 'East', '河南': 'Central', '湖北': 'Central', '湖南': 'Central', '广东': 'South', '广西': 'South', '海南': 'South', '香港': 'South', '澳门': 'South', '重庆': 'Southwest', '四川': 'Southwest', '贵州': 'Southwest', '云南': 'Southwest', '西藏': 'Southwest'}
province_maps = get_adm_maps(level='省')
classified_results = {'Northeast': [], 'North': [], 'Northwest': [], 'East': [], 'Central': [], 'South': [], 'Southwest': [], 'Unclassified': []}
for site_name, coords in my_sites.items():
    lon, lat = coords
    pt = Point(lon, lat)
    found_region = False
    for prov in province_maps:
        if prov['geometry'].contains(pt):
            prov_name = prov['province'] if 'province' in prov else prov['省/直辖市']
            for key_prov, region_name in REGION_MAPPING.items():
                if key_prov in prov_name:
                    classified_results[region_name].append(site_name)
                    found_region = True
                    break
            if found_region:
                break
    if not found_region:
        classified_results['Unclassified'].append(site_name)
        print(f'Unclassified radar: {site_name} (outside the available provincial polygons)')
print('\n' + '=' * 25 + ' Regional classification ' + '=' * 25)
for region, sites in classified_results.items():
    if sites:
        print(f"【{region}] Radar IDs: {', '.join(sites)}")


In [ ]:
case_num_region = {}
sample_num_region = {}
region_list = ['North', 'Northwest', 'East', 'Central', 'South', 'Southwest']
for setname in ['train', 'validation', 'test']:
    case_num_region[setname] = {}
    sample_num_region[setname] = {}
    for region_name in region_list:
        if len(classified_results[region_name]) != 0:
            case_num_region[setname][region_name] = 0
            sample_num_region[setname][region_name] = 0
for setname in ['train', 'validation', 'test']:
    split_matrix = load_pickle(CATALOG_ROOT / setname / 'train_dic_matrix.pkl')
    for case_id in split_matrix:
        stid = case_id.split('_')[-1]
        get = 0
        for region_ in region_list:
            if stid in classified_results[region_]:
                get = 1
                case_num_region[setname][region_] += 1
                sample_num_region[setname][region_] += len(split_matrix[case_id])
        if get == 0:
            print(setname, stid)


In [ ]:
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False
splits = ['train', 'validation', 'test']
splits_en = ['Training set', 'Validation set', 'Test set']
regions = ['North', 'Northwest', 'East', 'Central', 'South', 'Southwest']
REGION_COLORS = {'Northeast': '#CCCCCC', 'North': '#66B2FF', 'Northwest': '#99FF99', 'East': '#FFB266', 'Central': '#CC99FF', 'South': '#66FFFF', 'Southwest': '#FFFF66', 'Unclassified': '#CCCCCC'}
REGION_EN_LABELS = {'Northeast': 'Northeast China St.', 'North': 'North China St.', 'Northwest': 'Northwest China St.', 'East': 'East China St.', 'Central': 'Central China St.', 'South': 'South China St.', 'Southwest': 'Southwest China St.', 'Unclassified': 'Unclassified'}
fig = plt.figure(figsize=(18, 8), dpi=300)
gs_main = gridspec.GridSpec(1, 2, width_ratios=[1.2, 1.0], wspace=0.15)
ax_map = fig.add_subplot(gs_main[0], projection=ccrs.PlateCarree())
gs_pies = gridspec.GridSpecFromSubplotSpec(2, 3, subplot_spec=gs_main[1], hspace=0.3, wspace=0.15)
ax_map.set_extent([97, 123, 20, 43], crs=ccrs.PlateCarree())
province_maps = get_adm_maps(level='省')
for prov in province_maps:
    prov_name = prov['province'] if 'province' in prov else prov['省/直辖市']
    target_region = 'Unclassified'
    for key_prov, region_name in REGION_MAPPING.items():
        if key_prov in prov_name:
            target_region = region_name
            break
    facecolor = REGION_COLORS.get(target_region, '#FFFFFF')
    draw_map(prov['geometry'], ax=ax_map, facecolor=facecolor, edgecolor='gray', linewidth=0.5, alpha=0.4, zorder=1)
country_map = get_adm_maps(country='中华人民共和国', level='国')
draw_maps(country_map, ax=ax_map, linewidth=1.5, color='black', zorder=2)
for region, sites in classified_results.items():
    if not sites:
        continue
    lons = [my_sites[s][0] for s in sites]
    lats = [my_sites[s][1] for s in sites]
    en_label = REGION_EN_LABELS.get(region, region)
    site_count = len(sites)
    custom_label = f'× {site_count}    {en_label}'
    ax_map.scatter(lons, lats, color=REGION_COLORS[region], edgecolor='black', s=100, marker='o', label=custom_label, transform=ccrs.PlateCarree(), zorder=4)
gl = ax_map.gridlines(draw_labels=True, linestyle='--', alpha=0.5, color='gray')
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 13, 'family': 'Arial'}
gl.ylabel_style = {'size': 13, 'family': 'Arial'}
ax_map.legend(loc='upper left', prop={'family': 'Arial', 'size': 13}, handletextpad=-0.3)
ax_map.text(-0.08, 1.05, 'A', transform=ax_map.transAxes, ha='left', va='bottom', fontname='Arial', fontsize=23, fontweight='black')
ax_map.set_title('Spatial Distribution of Radar Stations', fontname='Arial', fontsize=18, fontweight='bold', pad=25)

def plot_pie_row(fig, gs, row_idx, data_dict, row_letter, row_title, title_loc):
    axes = []
    for col_idx, split in enumerate(splits):
        ax = fig.add_subplot(gs[row_idx, col_idx])
        axes.append(ax)
        values = [data_dict[split].get(reg, 0) for reg in regions]
        total_count = sum(values)
        colors = [REGION_COLORS[reg] for reg in regions]
        pie_radius = 1.1
        wedges, texts, autotexts = ax.pie(values, colors=colors, autopct=lambda p: f'{p:.1f}%' if p > 0 else '', startangle=140, radius=pie_radius, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
        for i, autotext in enumerate(autotexts):
            autotext.set_color('black')
            autotext.set_fontsize(14)
            autotext.set_fontname('Arial')
            if regions[i] == 'Northwest' and values[i] > 0:
                autotext.set_visible(False)
                theta = np.deg2rad((wedges[i].theta1 + wedges[i].theta2) / 2)
                x_edge = pie_radius * np.cos(theta)
                y_edge = pie_radius * np.sin(theta)
                x_text = (pie_radius + 0.3) * np.cos(theta)
                y_text = (pie_radius + 0.3) * np.sin(theta)
                pct_val = values[i] / total_count * 100
                ax.annotate(f'{pct_val:.1f}%', xy=(x_edge, y_edge), xytext=(x_text, y_text), ha='center', va='center', fontname='Arial', fontsize=14, arrowprops=dict(arrowstyle='-', color='black', lw=1.5))
        split_name = splits_en[col_idx]
        if row_title == 'Case Count':
            percent_str = round(100 * total_count / (1365 + 181 + 185), 2)
        else:
            percent_str = round(100 * total_count / (22249 + 2630 + 2657), 2)
        ax.set_title(f'{split_name}\n{total_count} ({percent_str}%)', fontname='Arial', fontsize=15, pad=0)
        if col_idx == 0:
            ax.text(-0.1, 1.25, row_letter, transform=ax.transAxes, ha='left', va='bottom', fontname='Arial', fontsize=23, fontweight='black')
        if col_idx == 1:
            ax.text(title_loc, 1.25, row_title, transform=ax.transAxes, ha='left', va='bottom', fontname='Arial', fontsize=18, fontweight='bold')
    return axes
plot_pie_row(fig, gs_pies, 0, case_num_region, 'B', 'Case Count', 0.22)
plot_pie_row(fig, gs_pies, 1, sample_num_region, 'C', 'Sample Count', 0.17)
plt.subplots_adjust(bottom=0.05, left=0.03, right=0.98, top=0.9)
plt.savefig(OUTPUT_ROOT / 'station_distribution.png', dpi=300, bbox_inches='tight')


In [ ]:
thre = 0.48
width = 1
kernel_size = 2 * width + 1
regions = ['Northeast', 'North', 'Northwest', 'East', 'Central', 'South', 'Southwest', 'Unclassified', 'All regions']
metrics_dict = {region: {'TP': torch.tensor(0, device=model.device, dtype=torch.int64), 'TN': torch.tensor(0, device=model.device, dtype=torch.int64), 'FP': torch.tensor(0, device=model.device, dtype=torch.int64), 'FN': torch.tensor(0, device=model.device, dtype=torch.int64)} for region in regions}
for case_id in tqdm(train_dic_matrix, total=len(train_dic_matrix), desc='Processing Cases'):
    prefix = case_id.split('_')[3]
    current_region = 'Unclassified'
    for region_ in classified_results:
        if prefix in classified_results[region_]:
            current_region = region_ if region_ in metrics_dict else 'Unclassified'
            break
    coords = np.array(list(mask_dic[case_id].values()))
    x_coords = torch.tensor(coords[:, 0], dtype=torch.long, device=model.device)
    y_coords = torch.tensor(coords[:, 1], dtype=torch.long, device=model.device)
    for index in train_dic_matrix[case_id]:
        input_data = np.load(train_dic_matrix[case_id][index]['input'])['arr_0']
        mask_data = np.load(train_dic_matrix[case_id][index]['mask'])['arr_0']
        output_data = np.load(train_dic_matrix[case_id][index]['output'])['arr_0']
        input_tensor = torch.tensor(input_data, dtype=torch.float32, device=model.device).unsqueeze(0)
        mask_tensor = torch.tensor(mask_data, dtype=torch.float32, device=model.device).unsqueeze(0).unsqueeze(0)
        output_tensor = torch.tensor(output_data, dtype=torch.float32, device=model.device).unsqueeze(0)
        with torch.no_grad():
            output_predict = model(input_tensor, mask_tensor)
            pooled_predict = F.max_pool2d(output_predict, kernel_size=kernel_size, stride=1, padding=width)
            pred_vals = pooled_predict[0, :, y_coords, x_coords]
            true_vals = output_tensor[0, :, y_coords, x_coords]
            pred_pos = pred_vals >= thre
            true_pos = true_vals >= 0.99
            step_TP = (pred_pos & true_pos).sum()
            step_FP = (pred_pos & ~true_pos).sum()
            step_FN = (~pred_pos & true_pos).sum()
            step_TN = (~pred_pos & ~true_pos).sum()
            metrics_dict[current_region]['TP'] += step_TP
            metrics_dict[current_region]['FP'] += step_FP
            metrics_dict[current_region]['FN'] += step_FN
            metrics_dict[current_region]['TN'] += step_TN
            metrics_dict['All regions']['TP'] += step_TP
            metrics_dict['All regions']['FP'] += step_FP
            metrics_dict['All regions']['FN'] += step_FN
            metrics_dict['All regions']['TN'] += step_TN
print(f'\nModel Path: {path}')
print('=' * 60)
region_metrics_data = {}
for region in regions:
    TP = metrics_dict[region]['TP'].item()
    TN = metrics_dict[region]['TN'].item()
    FP = metrics_dict[region]['FP'].item()
    FN = metrics_dict[region]['FN'].item()
    total_samples = TP + TN + FP + FN
    if total_samples == 0:
        continue
    print(f'【{region}] Station-time pairs: {total_samples}')
    print(f'  TP: {TP} | TN: {TN} | FP: {FP} | FN: {FN}')
    accuracy = (TP + TN) / total_samples if total_samples > 0 else 0
    precision = TP / (TP + FP) if TP + FP > 0 else 0
    recall = TP / (TP + FN) if TP + FN > 0 else 0
    denominator = TP + FP + FN
    ts = TP / denominator if denominator > 0 else 0
    r = (TP + FP) * (TP + FN) / total_samples if total_samples > 0 else 0
    ets = (TP - r) / (denominator - r) if denominator - r > 0 else 0
    print(f'  Accuracy:  {accuracy:.4f}')
    print(f'  Precision: {precision:.4f}')
    print(f'  Recall:    {recall:.4f}')
    print(f'  TS Score:  {ts:.4f}')
    print(f'  ETS Score: {ets:.4f}')
    print('-' * 40)
    region_metrics_data[region] = [precision, recall, ets]
pd.DataFrame.from_dict(region_metrics_data, orient='index', columns=['Precision', 'Recall', 'ETS']).to_csv(OUTPUT_ROOT / 'regional_metrics.csv', index_label='Region')


In [ ]:
thre = 0.48
width = 1
kernel_size = 2 * width + 1
metrics_dict = {'All regions': {'TP': torch.tensor(0, device=model.device, dtype=torch.int64), 'TN': torch.tensor(0, device=model.device, dtype=torch.int64), 'FP': torch.tensor(0, device=model.device, dtype=torch.int64), 'FN': torch.tensor(0, device=model.device, dtype=torch.int64)}}
for case_id in tqdm(train_dic_matrix, total=len(train_dic_matrix), desc='Processing Cases'):
    prefix = case_id.split('_')[3]
    if prefix not in metrics_dict:
        metrics_dict[prefix] = {'TP': torch.tensor(0, device=model.device, dtype=torch.int64), 'TN': torch.tensor(0, device=model.device, dtype=torch.int64), 'FP': torch.tensor(0, device=model.device, dtype=torch.int64), 'FN': torch.tensor(0, device=model.device, dtype=torch.int64)}
    coords = np.array(list(mask_dic[case_id].values()))
    x_coords = torch.tensor(coords[:, 0], dtype=torch.long, device=model.device)
    y_coords = torch.tensor(coords[:, 1], dtype=torch.long, device=model.device)
    for index in train_dic_matrix[case_id]:
        input_data = np.load(train_dic_matrix[case_id][index]['input'])['arr_0']
        mask_data = np.load(train_dic_matrix[case_id][index]['mask'])['arr_0']
        output_data = np.load(train_dic_matrix[case_id][index]['output'])['arr_0']
        input_tensor = torch.tensor(input_data, dtype=torch.float32, device=model.device).unsqueeze(0)
        mask_tensor = torch.tensor(mask_data, dtype=torch.float32, device=model.device).unsqueeze(0).unsqueeze(0)
        output_tensor = torch.tensor(output_data, dtype=torch.float32, device=model.device).unsqueeze(0)
        with torch.no_grad():
            output_predict = model(input_tensor, mask_tensor)
            pooled_predict = F.max_pool2d(output_predict, kernel_size=kernel_size, stride=1, padding=width)
            pred_vals = pooled_predict[0, :, y_coords, x_coords]
            true_vals = output_tensor[0, :, y_coords, x_coords]
            pred_pos = pred_vals >= thre
            true_pos = true_vals >= 0.99
            step_TP = (pred_pos & true_pos).sum()
            step_FP = (pred_pos & ~true_pos).sum()
            step_FN = (~pred_pos & true_pos).sum()
            step_TN = (~pred_pos & ~true_pos).sum()
            metrics_dict[prefix]['TP'] += step_TP
            metrics_dict[prefix]['FP'] += step_FP
            metrics_dict[prefix]['FN'] += step_FN
            metrics_dict[prefix]['TN'] += step_TN
            metrics_dict['All regions']['TP'] += step_TP
            metrics_dict['All regions']['FP'] += step_FP
            metrics_dict['All regions']['FN'] += step_FN
            metrics_dict['All regions']['TN'] += step_TN
print(f'\nModel Path: {path}')
print('=' * 60)
results_list = []
radar_stations = ['All regions'] + [k for k in metrics_dict.keys() if k != 'All regions']
for radar_id in radar_stations:
    TP = metrics_dict[radar_id]['TP'].item()
    TN = metrics_dict[radar_id]['TN'].item()
    FP = metrics_dict[radar_id]['FP'].item()
    FN = metrics_dict[radar_id]['FN'].item()
    total_samples = TP + TN + FP + FN
    if total_samples == 0:
        continue
    accuracy = (TP + TN) / total_samples if total_samples > 0 else 0
    precision = TP / (TP + FP) if TP + FP > 0 else 0
    recall = TP / (TP + FN) if TP + FN > 0 else 0
    denominator = TP + FP + FN
    ts = TP / denominator if denominator > 0 else 0
    r = (TP + FP) * (TP + FN) / total_samples if total_samples > 0 else 0
    ets = (TP - r) / (denominator - r) if denominator - r > 0 else 0
    results_list.append({'Radar_Station': radar_id, 'Samples': total_samples, 'TP': TP, 'TN': TN, 'FP': FP, 'FN': FN, 'Accuracy': round(accuracy, 4), 'Precision': round(precision, 4), 'Recall': round(recall, 4), 'TS_Score': round(ts, 4), 'ETS_Score': round(ets, 4)})
df_results = pd.DataFrame(results_list)
print(df_results.to_string(index=False))
csv_path = OUTPUT_ROOT / 'radar_evaluation_results.csv'
df_results.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f'\nResults saved: {csv_path}')
station_ets_scores = dict(zip(df_results['Radar_Station'], df_results['ETS_Score']))


In [ ]:
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False
metrics = ['Precision', 'Recall', 'ETS Score']
EN_TITLES = {'Northwest': 'Northwest China', 'North': 'North China', 'East': 'East China', 'Central': 'Central China', 'South': 'South China', 'Southwest': 'Southwest China'}
REGION_COLORS = {'Northeast': '#CCCCCC', 'North': '#66B2FF', 'Northwest': '#99FF99', 'East': '#FFB266', 'Central': '#CC99FF', 'South': '#66FFFF', 'Southwest': '#FFFF66', 'Unclassified': '#CCCCCC'}
region_centers = {'Northwest': (100.0, 39.0), 'North': (114.0, 39.5), 'Central': (112.5, 31.0), 'Southwest': (102.0, 26.0), 'South': (113.0, 24.0), 'East': (118.0, 28.5)}
fig = plt.figure(figsize=(16, 17), dpi=300)
gs = gridspec.GridSpec(4, 3, height_ratios=[1, 1.5, 1.5, 1], width_ratios=[1, 1, 1], hspace=0.3, wspace=0.25)
ax_map = fig.add_subplot(gs[1:3, 0:2], projection=ccrs.PlateCarree())
ax_map.set_extent([97, 123, 20, 43], crs=ccrs.PlateCarree())
province_maps = get_adm_maps(level='省')
for prov in province_maps:
    prov_name = prov['province'] if 'province' in prov else prov['省/直辖市']
    target_region = 'Unclassified'
    for key_prov, region_name in REGION_MAPPING.items():
        if key_prov in prov_name:
            target_region = region_name
            break
    facecolor = REGION_COLORS.get(target_region, '#FFFFFF')
    draw_map(prov['geometry'], ax=ax_map, facecolor=facecolor, edgecolor='gray', linewidth=0.5, alpha=0.4, zorder=1)
country_map = get_adm_maps(country='中华人民共和国', level='国')
draw_maps(country_map, ax=ax_map, linewidth=1.5, color='black', zorder=2)
all_lons = []
all_lats = []
all_ets = []
for region, sites in classified_results.items():
    if not sites:
        continue
    for s in sites:
        all_lons.append(my_sites[s][0])
        all_lats.append(my_sites[s][1])
        all_ets.append(station_ets_scores.get(s, np.nan))
sc = ax_map.scatter(all_lons, all_lats, c=all_ets, cmap='RdYlBu_r', edgecolor='black', s=200, marker='o', linewidths=1.2, transform=ccrs.PlateCarree(), zorder=4, vmin=0, vmax=0.3)
gl = ax_map.gridlines(draw_labels=True, linestyle='--', alpha=0.5, color='gray')
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 14, 'family': 'Arial'}
gl.ylabel_style = {'size': 14, 'family': 'Arial'}
ax_map.set_title('ETS Score of HailRU at Each Radar Station', fontname='Arial', fontsize=18, fontweight='bold', pad=10)
for spine in ax_map.spines.values():
    spine.set_linewidth(1.5)
    spine.set_edgecolor('black')
ax_map.text(-0.05, 1.02, 'D', transform=ax_map.transAxes, ha='left', va='bottom', fontname='Arial', fontsize=23, fontweight='black')
ax_cbar_container = fig.add_subplot(gs[1:3, 2])
ax_cbar_container.axis('off')
cbar_ax = ax_cbar_container.inset_axes([-0.0, 0.05, 0.08, 0.9])
cbar = plt.colorbar(sc, cax=cbar_ax, extend='both')
cbar.outline.set_linewidth(1.5)
cbar.ax.tick_params(labelsize=14, width=1.5)
cbar.set_label('ETS Score', fontname='Arial', fontsize=18, fontweight='bold', labelpad=15)

def plot_mini_bar(gs_pos, region_name, loc='top', letter=''):
    ax = fig.add_subplot(gs_pos)
    data = region_metrics_data.get(region_name)
    if data is None:
        ax.set_title(EN_TITLES[region_name])
        ax.text(0.5, 0.5, 'No evaluated samples', ha='center', va='center', transform=ax.transAxes)
        ax.set_axis_off()
        return ax
    region_color = REGION_COLORS[region_name]
    en_title = EN_TITLES[region_name]
    x = np.arange(len(metrics))
    bars = ax.bar(x, data, width=0.65, color=region_color, edgecolor='black', linewidth=0, alpha=1)
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width() / 2, height), xytext=(0, 2), textcoords='offset points', ha='center', va='bottom', fontsize=14, fontname='Arial')
    ax.set_xticks(x)
    ax.set_xticklabels(['Pre', 'Rec', 'ETS'], fontname='Arial', fontsize=14)
    ax.set_ylim(0, 0.35)
    ax.set_yticks([0.1, 0.2, 0.3])
    ax.tick_params(axis='x', width=1.5)
    ax.tick_params(axis='y', labelsize=13, width=1.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_edgecolor('black')
    ax.spines['bottom'].set_linewidth(1.5)
    ax.spines['left'].set_edgecolor('black')
    ax.spines['left'].set_linewidth(1.5)
    ax.plot(0, 1, transform=ax.transAxes, marker='^', markersize=7, color='black', clip_on=False)
    ax.set_title(en_title, fontname='Arial', fontsize=18, fontweight='bold', pad=10)
    if letter:
        ax.text(-0.1, 1.05, letter, transform=ax.transAxes, ha='left', va='bottom', fontname='Arial', fontsize=23, fontweight='black')
    return ax
ax_nw = plot_mini_bar(gs[0, 0], 'Northwest', loc='top', letter='A')
ax_n = plot_mini_bar(gs[0, 1], 'North', loc='top', letter='B')
ax_c = plot_mini_bar(gs[0, 2], 'Central', loc='top', letter='C')
ax_sw = plot_mini_bar(gs[3, 0], 'Southwest', loc='bottom', letter='E')
ax_s = plot_mini_bar(gs[3, 1], 'South', loc='bottom', letter='F')
ax_e = plot_mini_bar(gs[3, 2], 'East', loc='bottom', letter='G')
plt.subplots_adjust(bottom=0.06, top=0.94, left=0.04, right=0.96)
plt.savefig(OUTPUT_ROOT / 'regional_skill.png', dpi=300, bbox_inches='tight')
plt.show()
